<a href="https://colab.research.google.com/github/VeenusDadri/training/blob/master/advance_pyspark/join.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark Training').getOrCreate()
sc = spark.sparkContext

In [64]:
emp_df = spark.read.format('csv').options(delimiter=',', header=True).load('/content/emp.csv')

In [65]:
dept_df = spark.read.format('csv').options(delimiter=',', header=True).load('/content/dept.csv')

In [66]:
emp_df.show()


+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|          1|          101|     John Doe| 30|  Male| 50000|01-01-2015|
|          2|          101|   Jane Smith| 25|Female| 45000|15-02-2016|
|          3|          102|    Bob Brown| 35|  Male| 55000|01-05-2014|
|          4|          102|    Alice Lee| 28|Female| 48000|30-09-2017|
|          5|          103|    Jack Chan| 40|  Male| 60000|01-04-2013|
|          6|          103|    Jill Wong| 32|Female| 52000|01-07-2018|
|          7|          101|James Johnson| 42|  Male| 70000|15-03-2012|
|          8|          102|     Kate Kim| 29|Female| 51000|01-10-2019|
|          9|          103|      Tom Tan| 33|  Male| 58000|01-06-2016|
|         10|          104|     Lisa Lee| 27|Female| 47000|01-08-2018|
|         11|          104|   David Park| 38|  Male| 65000|01-11-2015|
|     

In [67]:
dept_df.show()

+-------------+--------------------+-------+---------+-------+
|department_id|     department_name|   city|  country| budget|
+-------------+--------------------+-------+---------+-------+
|          101|               Sales|    NYC|       US|1000000|
|          102|           Marketing|     LA|       US| 900000|
|          103|             Finance| London|       UK|1200000|
|          104|         Engineering|Beijing|    China|1500000|
|          105|     Human Resources|  Tokyo|    Japan| 800000|
|          106|Research and Deve...|  Perth|Australia|1100000|
|          107|    Customer Service| Sydney|Australia| 950000|
+-------------+--------------------+-------+---------+-------+



In [68]:
emp_df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [69]:
dept_df.printSchema()

root
 |-- department_id: string (nullable = true)
 |-- department_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- budget: string (nullable = true)



In [70]:
emp_df.rdd.getNumPartitions()

1

In [71]:
emp_partitioned=emp_df.repartition(4)
emp_partitioned.rdd.getNumPartitions()

4

In [72]:
from pyspark.sql.functions import *
emp_partitioned.withColumn('partition_id', spark_partition_id()).show()

+-----------+-------------+-------------+---+------+------+----------+------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|partition_id|
+-----------+-------------+-------------+---+------+------+----------+------------+
|         11|          104|   David Park| 38|  Male| 65000|01-11-2015|           0|
|         10|          104|     Lisa Lee| 27|Female| 47000|01-08-2018|           0|
|         19|          103|  Steven Chen| 36|  Male| 62000|01-08-2015|           0|
|          9|          103|      Tom Tan| 33|  Male| 58000|01-06-2016|           0|
|         13|          106|    Brian Kim| 45|  Male| 75000|01-07-2011|           0|
|          1|          101|     John Doe| 30|  Male| 50000|01-01-2015|           1|
|          4|          102|    Alice Lee| 28|Female| 48000|30-09-2017|           1|
|         17|          105|  George Wang| 34|  Male| 57000|15-03-2016|           1|
|         20|          102|    Grace Kim| 32|Female| 53000|01-11-2018|      

In [73]:
emp_repartitioned= emp_df.repartition(4, "department_id").withColumn('partition_id', spark_partition_id())
emp_repartitioned.show()

+-----------+-------------+-------------+---+------+------+----------+------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|partition_id|
+-----------+-------------+-------------+---+------+------+----------+------------+
|          3|          102|    Bob Brown| 35|  Male| 55000|01-05-2014|           0|
|          4|          102|    Alice Lee| 28|Female| 48000|30-09-2017|           0|
|          8|          102|     Kate Kim| 29|Female| 51000|01-10-2019|           0|
|         14|          107|    Emily Lee| 26|Female| 46000|01-01-2019|           0|
|         16|          107|  Kelly Zhang| 30|Female| 49000|01-04-2018|           0|
|         20|          102|    Grace Kim| 32|Female| 53000|01-11-2018|           0|
|         12|          105|   Susan Chen| 31|Female| 54000|15-02-2017|           1|
|         17|          105|  George Wang| 34|  Male| 57000|15-03-2016|           1|
|         10|          104|     Lisa Lee| 27|Female| 47000|01-08-2018|      

In [74]:
emp_repartitioned.groupBy("partition_id").count().show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|    6|
|           1|    2|
|           2|    5|
|           3|    7|
+------------+-----+



In [75]:
emp_dept= emp_df.alias("e").join(dept_df.alias("d"), how= "inner", on='department_id')
print(emp_dept.count())
emp_dept.show()

20
+-------------+-----------+-------------+---+------+------+----------+--------------------+-------+---------+-------+
|department_id|employee_id|         name|age|gender|salary| hire_date|     department_name|   city|  country| budget|
+-------------+-----------+-------------+---+------+------+----------+--------------------+-------+---------+-------+
|          101|          1|     John Doe| 30|  Male| 50000|01-01-2015|               Sales|    NYC|       US|1000000|
|          101|          2|   Jane Smith| 25|Female| 45000|15-02-2016|               Sales|    NYC|       US|1000000|
|          102|          3|    Bob Brown| 35|  Male| 55000|01-05-2014|           Marketing|     LA|       US| 900000|
|          102|          4|    Alice Lee| 28|Female| 48000|30-09-2017|           Marketing|     LA|       US| 900000|
|          103|          5|    Jack Chan| 40|  Male| 60000|01-04-2013|             Finance| London|       UK|1200000|
|          103|          6|    Jill Wong| 32|Female| 

In [76]:
emp_dept.select('department_id','name','salary', 'department_name').show()

+-------------+-------------+------+--------------------+
|department_id|         name|salary|     department_name|
+-------------+-------------+------+--------------------+
|          101|     John Doe| 50000|               Sales|
|          101|   Jane Smith| 45000|               Sales|
|          102|    Bob Brown| 55000|           Marketing|
|          102|    Alice Lee| 48000|           Marketing|
|          103|    Jack Chan| 60000|             Finance|
|          103|    Jill Wong| 52000|             Finance|
|          101|James Johnson| 70000|               Sales|
|          102|     Kate Kim| 51000|           Marketing|
|          103|      Tom Tan| 58000|             Finance|
|          104|     Lisa Lee| 47000|         Engineering|
|          104|   David Park| 65000|         Engineering|
|          105|   Susan Chen| 54000|     Human Resources|
|          106|    Brian Kim| 75000|Research and Deve...|
|          107|    Emily Lee| 46000|    Customer Service|
|          106

In [77]:
e_d= emp_df.alias("e").join(dept_df.alias("d"), how= "left_outer", on='department_id')
print(e_d.count())
e_d.show()

20
+-------------+-----------+-------------+---+------+------+----------+--------------------+-------+---------+-------+
|department_id|employee_id|         name|age|gender|salary| hire_date|     department_name|   city|  country| budget|
+-------------+-----------+-------------+---+------+------+----------+--------------------+-------+---------+-------+
|          101|          1|     John Doe| 30|  Male| 50000|01-01-2015|               Sales|    NYC|       US|1000000|
|          101|          2|   Jane Smith| 25|Female| 45000|15-02-2016|               Sales|    NYC|       US|1000000|
|          102|          3|    Bob Brown| 35|  Male| 55000|01-05-2014|           Marketing|     LA|       US| 900000|
|          102|          4|    Alice Lee| 28|Female| 48000|30-09-2017|           Marketing|     LA|       US| 900000|
|          103|          5|    Jack Chan| 40|  Male| 60000|01-04-2013|             Finance| London|       UK|1200000|
|          103|          6|    Jill Wong| 32|Female| 

In [78]:
e_d.select('department_id','name','salary', 'department_name').show()

+-------------+-------------+------+--------------------+
|department_id|         name|salary|     department_name|
+-------------+-------------+------+--------------------+
|          101|     John Doe| 50000|               Sales|
|          101|   Jane Smith| 45000|               Sales|
|          102|    Bob Brown| 55000|           Marketing|
|          102|    Alice Lee| 48000|           Marketing|
|          103|    Jack Chan| 60000|             Finance|
|          103|    Jill Wong| 52000|             Finance|
|          101|James Johnson| 70000|               Sales|
|          102|     Kate Kim| 51000|           Marketing|
|          103|      Tom Tan| 58000|             Finance|
|          104|     Lisa Lee| 47000|         Engineering|
|          104|   David Park| 65000|         Engineering|
|          105|   Susan Chen| 54000|     Human Resources|
|          106|    Brian Kim| 75000|Research and Deve...|
|          107|    Emily Lee| 46000|    Customer Service|
|          106

In [84]:
df1= emp_df.join(dept_df, how= "left_outer", on=(emp_df.department_id==dept_df.department_id) & ((emp_df.department_id=='102') | (emp_df.department_id=='101')))
df1.show()

+-----------+-------------+-------------+---+------+------+----------+-------------+---------------+----+-------+-------+
|employee_id|department_id|         name|age|gender|salary| hire_date|department_id|department_name|city|country| budget|
+-----------+-------------+-------------+---+------+------+----------+-------------+---------------+----+-------+-------+
|          1|          101|     John Doe| 30|  Male| 50000|01-01-2015|          101|          Sales| NYC|     US|1000000|
|          2|          101|   Jane Smith| 25|Female| 45000|15-02-2016|          101|          Sales| NYC|     US|1000000|
|          3|          102|    Bob Brown| 35|  Male| 55000|01-05-2014|          102|      Marketing|  LA|     US| 900000|
|          4|          102|    Alice Lee| 28|Female| 48000|30-09-2017|          102|      Marketing|  LA|     US| 900000|
|          5|          103|    Jack Chan| 40|  Male| 60000|01-04-2013|         NULL|           NULL|NULL|   NULL|   NULL|
|          6|          1